In [2]:
import os
import shutil
from transformers import AutoTokenizer

def compare_and_cleanup_tokenizers(local_experiment_path: str, hf_model_name: str, save_path: str = "./klue_bert_base_tokenizer"):
    print(f"1. 허브 모델 로드 및 로컬 저장 진행 중...")
    print(f" - 허브 모델: {hf_model_name}")
    print(f" - 임시 저장 위치: {save_path}")
    
    # 허브에서 다운로드 및 지정한 로컬 경로에 저장
    klue_tokenizer_hub = AutoTokenizer.from_pretrained(hf_model_name)
    klue_tokenizer_hub.save_pretrained(save_path)
    
    try:
        print(f"\n2. 비교를 위해 로컬 경로에서 토크나이저 로드 중...")
        local_tokenizer = AutoTokenizer.from_pretrained(local_experiment_path)
        klue_tokenizer_local = AutoTokenizer.from_pretrained(save_path)

        # 3. 어휘 사전(Vocabulary) 데이터 추출
        local_vocab = local_tokenizer.get_vocab()
        klue_vocab = klue_tokenizer_local.get_vocab()

        print(f"\n3. 어휘 사전(Vocabulary) 크기 확인")
        print(f" - 실험용 토크나이저 Vocab 크기: {len(local_vocab)}")
        print(f" - 공식 KLUE 토크나이저 Vocab 크기: {len(klue_vocab)}")

        # 4. 전수조사
        is_identical = local_vocab == klue_vocab

        if is_identical:
            print("\n[최종 결론]")
            print("실험 중인 토크나이저의 어휘 사전은 klue/bert-base와 완벽하게 일치합니다.")
            return

        print("\n[최종 결론]")
        print("실험 중인 토크나이저와 klue/bert-base의 어휘 사전이 다릅니다.")
        
        # 상세 차이 분석
        local_tokens = set(local_vocab.keys())
        klue_tokens = set(klue_vocab.keys())
        
        only_in_local = local_tokens - klue_tokens
        only_in_klue = klue_tokens - local_tokens
        
        print(f"\n[상세 분석 결과]")
        print(f" - 실험용 모델에만 존재하는 토큰 개수: {len(only_in_local)}")
        print(f" - 공식 KLUE에만 존재하는 토큰 개수: {len(only_in_klue)}")
        
        if only_in_local:
            print(f" - 실험용 모델 전용 토큰 예시: {list(only_in_local)[:5]}")
        if only_in_klue:
            print(f" - 공식 KLUE 전용 토큰 예시: {list(only_in_klue)[:5]}")
            
        common_tokens = local_tokens & klue_tokens
        diff_id_tokens = {token for token in common_tokens if local_vocab[token] != klue_vocab[token]}
        
        print(f" - 토큰은 같으나 할당된 ID가 다른 토큰 개수: {len(diff_id_tokens)}")
        if diff_id_tokens:
            print(f" - ID 매핑이 다른 토큰 예시: {list(diff_id_tokens)[:5]}")

    finally:
        # 5. 임시 디렉토리 정리 (오류 발생이나 return 여부와 무관하게 항상 실행됨)
        if os.path.exists(save_path):
            shutil.rmtree(save_path)
            print(f"\n[정리 완료] 실험을 위해 임시로 저장했던 '{save_path}' 폴더와 내부 파일들을 모두 삭제했습니다.")

# 함수 실행
compare_and_cleanup_tokenizers("./tokenizer/", "klue/bert-base")

1. 허브 모델 로드 및 로컬 저장 진행 중...
 - 허브 모델: klue/bert-base
 - 임시 저장 위치: ./klue_bert_base_tokenizer



2. 비교를 위해 로컬 경로에서 토크나이저 로드 중...

3. 어휘 사전(Vocabulary) 크기 확인
 - 실험용 토크나이저 Vocab 크기: 32000
 - 공식 KLUE 토크나이저 Vocab 크기: 32000

[최종 결론]
실험 중인 토크나이저와 klue/bert-base의 어휘 사전이 다릅니다.

[상세 분석 결과]
 - 실험용 모델에만 존재하는 토큰 개수: 23522
 - 공식 KLUE에만 존재하는 토큰 개수: 23522
 - 실험용 모델 전용 토큰 예시: ['알레르겐', '기술하고자', '揮', '설문조사와', '##S는']
 - 공식 KLUE 전용 토큰 예시: ['민변', '건물', '학사', '수줍', '덕분']
 - 토큰은 같으나 할당된 ID가 다른 토큰 개수: 8414
 - ID 매핑이 다른 토큰 예시: ['192', '보아', '인해', '##염', '179']

[정리 완료] 실험을 위해 임시로 저장했던 './klue_bert_base_tokenizer' 폴더와 내부 파일들을 모두 삭제했습니다.
